# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Transparent rule first. Week-5 models must beat this on the same metric.


## 1. My rule and its reason codes

**Rule (plain words):** Score pages higher when they are visible (impressions) AND stale
(days since update), with a boost when CTR is weak on a still-ranked page. Emit one primary
reason code and an action label.

**Signal checks (flag-linked):** (1) staleness behind refresh flags, (2) CTR-vs-visibility
behind CTR-fix logic.

**Reason codes:** `stale_visible_page`, `low_ctr_visible_page`, `general_refresh_review`

**Actions:** `refresh`, `refresh_and_review_ctr`, `monitor`


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ROOT = Path('.').resolve()
# Support running from repo root or work/notebooks
if (ROOT / 'data/raw/content_refresh_anonymized.csv').exists():
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
elif (ROOT.parents[1] / 'data/raw/content_refresh_anonymized.csv').exists():
    ROOT = ROOT.parents[1]
    DATA = ROOT / 'data/raw/content_refresh_anonymized.csv'
else:
    # Colab / nested
    for cand in [ROOT, *ROOT.parents]:
        if (cand / 'data/raw/content_refresh_anonymized.csv').exists():
            ROOT = cand
            DATA = cand / 'data/raw/content_refresh_anonymized.csv'
            break
    else:
        raise FileNotFoundError('data/raw/content_refresh_anonymized.csv')

WORK_OUT = ROOT / 'work' / 'outputs'
WORK_OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA)
print(f'Loaded {len(df):,} rows × {df.shape[1]} cols from {DATA.relative_to(ROOT)}')
print(f'Clients (pseudonymous): {df["client_id"].nunique()}')

df['label'] = (df['trend_direction'].str.lower()=='down').astype(int)

def show_buckets(title, series_bool):
    tab = df.assign(bucket=np.where(series_bool, 'HI', 'LO')).groupby('bucket')['label'].agg(['mean','count'])
    print(f'\n{title}')
    print(tab)
    hi, lo = tab.loc['HI','mean'] if 'HI' in tab.index else 0, tab.loc['LO','mean'] if 'LO' in tab.index else 0
    v = 'CONFIRMED' if hi > lo + 0.02 else ('OPPOSITE' if lo > hi + 0.02 else 'MIXED')
    print('Verdict:', v, '| n_HI=', int(series_bool.sum()))
    return v

v_stale = show_buckets('Signal 1: days_since_last_update ≥ 180 (flag-linked staleness)',
                       df['days_since_last_update'] >= 180)
v_ctr = show_buckets('Signal 2: impressions≥500 & 0<pos≤20 & ctr<0.5 (CTR-fix linked)',
                     (df['impressions_90d']>=500) & (df['avg_position']>0) & (df['avg_position']<=20) & (df['ctr']<0.5))


Loaded 30,000 rows × 44 cols from data\raw\content_refresh_anonymized.csv
Clients (pseudonymous): 32

Signal 1: days_since_last_update ≥ 180 (flag-linked staleness)
            mean  count
bucket                 
HI      0.471264    174
LO      0.542480  29826
Verdict: OPPOSITE | n_HI= 174

Signal 2: impressions≥500 & 0<pos≤20 & ctr<0.5 (CTR-fix linked)
            mean  count
bucket                 
HI      0.627113   9759
LO      0.501062  20241
Verdict: CONFIRMED | n_HI= 9759


## 2. Build the ranked queue (writes the CSV)

One score, one primary reason code, one action. Writes `work/outputs/baseline_action_score.csv`
(gitignored by design). Metrics JSON is committed as the receipt.


In [2]:
def primary_reason(row):
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return 'stale_visible_page'
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        return 'low_ctr_visible_page'
    return 'general_refresh_review'

def action_for(reason):
    return {
        'stale_visible_page': 'refresh',
        'low_ctr_visible_page': 'refresh_and_review_ctr',
        'general_refresh_review': 'monitor',
    }[reason]

# Transparent score in [0,1]
vis = df['impressions_90d'].rank(pct=True)
fresh_risk = df['days_since_last_update'].rank(pct=True)
ctr_gap = ((df['impressions_90d']>=500) & (df['avg_position']>0) & (df['avg_position']<=20) & (df['ctr']<0.5)).astype(float)
score = (0.45 * vis + 0.40 * fresh_risk + 0.15 * ctr_gap).clip(0, 1)

out = df[['content_id','client_id','impressions_90d','days_since_last_update','avg_position','ctr','label']].copy()
out['baseline_score'] = score
out['reason_code'] = df.apply(primary_reason, axis=1)
out['action'] = out['reason_code'].map(action_for)
out['baseline_rank'] = out['baseline_score'].rank(method='first', ascending=False).astype(int)
out = out.sort_values('baseline_rank')

csv_path = WORK_OUT / 'baseline_action_score.csv'
out.to_csv(csv_path, index=False)
# Receipt metrics (commit this)
top10_prec = float(out.head(10)['label'].mean())
top50_prec = float(out.head(50)['label'].mean())
receipt = {
    'n': int(len(out)),
    'precision_at_10_full_data': top10_prec,
    'precision_at_50_full_data': top50_prec,
    'base_rate': float(out['label'].mean()),
    'reason_mix': out['reason_code'].value_counts().to_dict(),
    'action_mix': out['action'].value_counts().to_dict(),
    'signal_verdicts': {'staleness': v_stale, 'ctr_visible': v_ctr},
    'note': 'Full-data Precision@K is descriptive; model comparison uses client-holdout.',
}
(WORK_OUT / 'baseline_action_score_metrics.json').write_text(json.dumps(receipt, indent=2))
print('Wrote', csv_path.relative_to(ROOT))
print(json.dumps(receipt, indent=2))


Wrote work\outputs\baseline_action_score.csv
{
  "n": 30000,
  "precision_at_10_full_data": 0.7,
  "precision_at_50_full_data": 0.42,
  "base_rate": 0.5420666666666667,
  "reason_mix": {
    "general_refresh_review": 20234,
    "low_ctr_visible_page": 9749,
    "stale_visible_page": 17
  },
  "action_mix": {
    "monitor": 20234,
    "refresh_and_review_ctr": 9749,
    "refresh": 17
  },
  "signal_verdicts": {
    "staleness": "OPPOSITE",
    "ctr_visible": "CONFIRMED"
  },
  "note": "Full-data Precision@K is descriptive; model comparison uses client-holdout."
}


## 3. Top-10 review

For each of the top ten: action, why it is there, and what would make it wrong.


In [3]:
top10 = out.head(10).copy()
rows = []
for _, r in top10.iterrows():
    wrong = {
        'stale_visible_page': 'Page was intentionally frozen (legal/seasonal) or impressions are bot-inflated.',
        'low_ctr_visible_page': 'Low CTR is expected for the query intent (navigational) or position is mismeasured (avg_position quirk).',
        'general_refresh_review': 'Score driven by volume alone without a real content problem.',
    }[r['reason_code']]
    rows.append({
        'rank': int(r['baseline_rank']),
        'action': r['action'],
        'reason': r['reason_code'],
        'score': round(float(r['baseline_score']), 3),
        'declining_label': int(r['label']),
        'what_would_make_it_wrong': wrong,
    })
review = pd.DataFrame(rows)
print(review.to_string(index=False))
(WORK_OUT / 'baseline_top10_review.json').write_text(json.dumps(rows, indent=2))


 rank                 action               reason  score  declining_label                                                                                 what_would_make_it_wrong
    1 refresh_and_review_ctr low_ctr_visible_page  0.993                0 Low CTR is expected for the query intent (navigational) or position is mismeasured (avg_position quirk).
    2                refresh   stale_visible_page  0.992                1                          Page was intentionally frozen (legal/seasonal) or impressions are bot-inflated.
    3 refresh_and_review_ctr low_ctr_visible_page  0.973                1 Low CTR is expected for the query intent (navigational) or position is mismeasured (avg_position quirk).
    4                refresh   stale_visible_page  0.957                1                          Page was intentionally frozen (legal/seasonal) or impressions are bot-inflated.
    5 refresh_and_review_ctr low_ctr_visible_page  0.940                1 Low CTR is expected for the que

2788

## 4. Weak picks + leakage check

Weak picks tend to be high-impression pages that are not actually stale or declining.
Confirm no `trend_direction` / `trend_pct` entered the score.


In [4]:
weak = out.head(50)
weak_stable = weak[weak['label']==0]
print('Among top-50, stable (label=0) count:', len(weak_stable))
if len(weak_stable):
    print(weak_stable[['baseline_rank','reason_code','action','impressions_90d','days_since_last_update']].head(5).to_string(index=False))
print('\nLeakage check: score formula uses only impressions, days_since_last_update, avg_position, ctr.')
print('trend_direction/trend_pct not used. Label used only for evaluation, not scoring.')


Among top-50, stable (label=0) count: 29
 baseline_rank          reason_code                 action  impressions_90d  days_since_last_update
             1 low_ctr_visible_page refresh_and_review_ctr            79035                     106
             9 low_ctr_visible_page refresh_and_review_ctr           295097                     104
            10 low_ctr_visible_page refresh_and_review_ctr           211366                     104
            12 low_ctr_visible_page refresh_and_review_ctr           201584                     104
            13 low_ctr_visible_page refresh_and_review_ctr           201111                     104

Leakage check: score formula uses only impressions, days_since_last_update, avg_position, ctr.
trend_direction/trend_pct not used. Label used only for evaluation, not scoring.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
